# E-Commerce Behaviour Pipeline
## Big Data Project — Group A7

Multi-stage distributed data pipeline processing **~14.6 GB** of e-commerce event logs using **Dask** and **Apache Spark**.

| | |
|---|---|
| **Dataset** | [Kaggle — eCommerce behavior data from multi-category store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store) |
| **Files** | `2019-Oct.csv` (5.6 GB) · `2019-Nov.csv` (9.0 GB) |
| **Schema** | `event_time`, `event_type`, `product_id`, `category_id`, `category_code`, `brand`, `price`, `user_id`, `user_session` |

### Pipeline Stages
1. **Ingestion & Validation** — load CSVs, schema enforcement, null / range checks, cleaning
2. **Dask Processing** — `filter`, `aggregate`, `pivot`, `assign`, advanced `groupby` (5 analyses)
3. **Spark Processing** — same questions via PySpark + window function, reads **raw CSV directly**
4. **Storage** — intermediate + final results written as **Parquet**
5. **Framework Comparison** — timing and ergonomics summary

In [1]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '..')   # make src/ importable from notebooks/

import dask
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
from pathlib import Path
import pandas as pd

from src.config import (
    PROJECT_ROOT, DATA_DIR, OUTPUT_DIR, LOGS_DIR,
    RAW_OCT, RAW_NOV,
    PARQUET_VALIDATED, RESULTS_DIR,
    VALID_EVENT_TYPES, DTYPES,
)
from src.logger     import StructuredLogger
from src.validation import validate_schema, data_quality_report, clean_data
from src.ingestion  import load_csv, load_csvs
from src.transformations_dask import (
    compute_revenue_by_category,
    compute_conversion_funnel,
    compute_hourly_activity,
    compute_session_stats,
    compute_top_brands,
)
from src.storage import save_parquet_dask, save_parquet_pandas, load_parquet

try:
    from pyspark.sql import SparkSession
    from src.transformations_spark import (
        get_spark_session, get_spark_dashboard_url,
        load_csv_spark,
        compute_revenue_by_category_spark,
        compute_conversion_funnel_spark,
        compute_window_rank_spark,
        compute_hourly_activity_spark,
    )
    SPARK_AVAILABLE = True
except ImportError:
    SPARK_AVAILABLE = False
    print('PySpark not found — Spark stage will be skipped.')

log = StructuredLogger('pipeline')
timings: dict[str, float] = {}

for d in [OUTPUT_DIR, LOGS_DIR, PARQUET_VALIDATED, RESULTS_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Output dir   : {OUTPUT_DIR}')
print(f'Spark        : {SPARK_AVAILABLE}')

Project root : F:\Group Project-A7\Group-A7-Project
Data dir     : F:\Group Project-A7\Group-A7-Project\data
Output dir   : F:\Group Project-A7\Group-A7-Project\output
Spark        : True


---
## Stage 1 — Data Ingestion & Validation

**Operations:** load two CSVs → schema check → null / value-range assertions → row-level cleaning  
**Error handling:** `on_bad_lines='skip'` drops malformed CSV rows; retry logic handles transient I/O failures; `RuntimeError` raised on schema mismatch.

> **Note:** `load_csvs()` passes both file paths directly to `dd.read_csv` as a list — Dask handles this natively without any concat or merge pass. All computation is **lazy** until `save_parquet_dask()` triggers the first full read.

In [2]:
log.info('pipeline_started')

# ── Load both CSVs in one lazy read (no merge/concat pass) ───────────────────
with log.timer('ingestion') as t:
    raw_ddf = load_csvs([RAW_OCT, RAW_NOV])
timings['ingestion'] = t.elapsed

print(f'Partitions : {raw_ddf.npartitions}')
print(f'Columns    : {list(raw_ddf.columns)}')

# ── Schema validation (fast — no compute triggered) ──────────────────────────
schema = validate_schema(raw_ddf)
print(f'\nSchema valid : {schema["valid"]}')
if not schema['valid']:
    raise RuntimeError(f'Missing columns: {schema["missing"]}')

raw_ddf.head(3)

{"ts": "2026-03-13T10:02:02.987003+00:00", "level": "INFO", "event": "pipeline_started"}
{"ts": "2026-03-13T10:02:02.988002+00:00", "level": "INFO", "event": "ingestion_started"}
{"ts": "2026-03-13T10:02:02.988505+00:00", "level": "INFO", "event": "load_csvs_started", "files": ["F:\\Group Project-A7\\Group-A7-Project\\data\\2019-Oct.csv", "F:\\Group Project-A7\\Group-A7-Project\\data\\2019-Nov.csv"]}
{"ts": "2026-03-13T10:02:03.069414+00:00", "level": "INFO", "event": "load_csvs_ok", "files": 2, "partitions": 114}
{"ts": "2026-03-13T10:02:03.070420+00:00", "level": "INFO", "event": "ingestion_completed", "elapsed_s": 0.08}
{"ts": "2026-03-13T10:02:03.070420+00:00", "level": "INFO", "event": "schema_valid", "columns": ["brand", "category_code", "category_id", "event_time", "event_type", "price", "product_id", "user_id", "user_session"]}


Partitions : 114
Columns    : ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']

Schema valid : True


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00+00:00,view,44600062,2103807459595387724,<NA>,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00+00:00,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01+00:00,view,17200506,2053013559792632471,furniture.living_room.sofa,<NA>,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8


In [3]:
# ── Clean (lazy — still no compute) ──────────────────────────────────────────
with log.timer('cleaning') as t:
    clean_ddf = clean_data(raw_ddf)
timings['cleaning'] = t.elapsed

# ── Persist validated data as Parquet, partitioned by event_type ──────────────
# This is the first cell that triggers real computation:
#   read 14.6 GB CSV → filter → write Parquet. Expect 10-30 min on a laptop.
print('Writing validated Parquet — this triggers the first full read of the CSV files.')
print('Expected time: 10-30 minutes depending on disk speed.\n')
with log.timer('save_validated') as t:
    save_parquet_dask(clean_ddf, PARQUET_VALIDATED, partition_on=['event_type'])
timings['save_validated'] = t.elapsed

print(f'\nValidated Parquet → {PARQUET_VALIDATED}')
print(f'Elapsed           : {timings["save_validated"]:.1f}s')

# Reload from Parquet — all Dask analyses read this instead of the raw CSVs
validated_ddf = load_parquet(PARQUET_VALIDATED)
print(f'Reloaded partitions: {validated_ddf.npartitions}')

{"ts": "2026-03-13T10:02:07.908075+00:00", "level": "INFO", "event": "cleaning_started"}
{"ts": "2026-03-13T10:02:07.916602+00:00", "level": "INFO", "event": "cleaning_rules_applied"}
{"ts": "2026-03-13T10:02:07.917604+00:00", "level": "INFO", "event": "cleaning_completed", "elapsed_s": 0.01}
{"ts": "2026-03-13T10:02:07.917604+00:00", "level": "INFO", "event": "save_validated_started"}
{"ts": "2026-03-13T10:02:07.919107+00:00", "level": "INFO", "event": "saving_parquet_dask", "path": "F:\\Group Project-A7\\Group-A7-Project\\output\\parquet\\validated"}


Writing validated Parquet — this triggers the first full read of the CSV files.
Expected time: 10-30 minutes depending on disk speed.



{"ts": "2026-03-13T10:06:41.695220+00:00", "level": "INFO", "event": "saved_parquet_dask", "path": "F:\\Group Project-A7\\Group-A7-Project\\output\\parquet\\validated"}
{"ts": "2026-03-13T10:06:41.696724+00:00", "level": "INFO", "event": "save_validated_completed", "elapsed_s": 273.78}



Validated Parquet → F:\Group Project-A7\Group-A7-Project\output\parquet\validated
Elapsed           : 273.8s
Reloaded partitions: 332


---
## Stage 1.5 — Quick Testing with Sample Data

**Purpose:** Verify Dask and Spark pipeline logic on a small subset before running full-scale processing.

**Operations:** sample ~1% of validated data → run the same 5 Dask analyses → run equivalent Spark analyses  
**Benefits:** quick iteration, framework ergonomics comparison, early error detection without 20+ min wait time.

> All analyses use the same transformation functions as Stages 2 & 3, ensuring consistency.

In [6]:
# ── Create sample dataset (~1% of validated data) ────────────────────────────
sample_fraction = 0.01
print(f'Creating sample dataset ({sample_fraction*100:.1f}% of validated data)...')

with log.timer('sample_creation') as t:
    sample_ddf = validated_ddf.sample(frac=sample_fraction, random_state=42)
    sample_ddf_computed = sample_ddf.compute()
timings['sample_creation'] = t.elapsed

print(f'  Original rows      : {len(validated_ddf)}')
print(f'  Sample rows        : {len(sample_ddf_computed):,}')
print(f'  Time to create     : {timings["sample_creation"]:.1f}s\n')

# ── Convert sample back to Dask for processing ──────────────────────────────
import dask.dataframe as dd
sample_dask = dd.from_pandas(sample_ddf_computed, npartitions=4)

# ── Quick Dask tests on sample ──────────────────────────────────────────────
print('=== DASK Tests on Sample ===')
sample_dask_results = {}

with LocalCluster(n_workers=2, threads_per_worker=2, memory_limit='2GB') as cluster:
    with Client(cluster) as client:
        # 1 — Revenue by category (sample)
        with log.timer('sample_dask_revenue') as t:
            sample_dask_results['revenue'] = compute_revenue_by_category(sample_dask)
        timings['sample_dask_revenue'] = t.elapsed
        print(f'[✓] Revenue by category       : {timings["sample_dask_revenue"]:.2f}s')
        print(f'    Top 5: {sample_dask_results["revenue"].head(5)["top_category"].tolist()}\n')

        # 2 — Conversion funnel (sample)
        with log.timer('sample_dask_funnel') as t:
            sample_dask_results['funnel'] = compute_conversion_funnel(sample_dask)
        timings['sample_dask_funnel'] = t.elapsed
        print(f'[✓] Conversion funnel         : {timings["sample_dask_funnel"]:.2f}s')

        # 3 — Hourly activity (sample)
        with log.timer('sample_dask_hourly') as t:
            sample_dask_results['hourly'] = compute_hourly_activity(sample_dask)
        timings['sample_dask_hourly'] = t.elapsed
        print(f'[✓] Hourly activity           : {timings["sample_dask_hourly"]:.2f}s')

        # 5 — Top brands (sample)
        with log.timer('sample_dask_brands') as t:
            sample_dask_results['brands'] = compute_top_brands(sample_dask, top_n=10)
        timings['sample_dask_brands'] = t.elapsed
        print(f'[✓] Top brands                : {timings["sample_dask_brands"]:.2f}s\n')

# 4 — Session stats (sample, no distributed client)
with log.timer('sample_dask_sessions') as t:
    sample_dask_results['sessions'] = compute_session_stats(sample_dask)
timings['sample_dask_sessions'] = t.elapsed
print(f'[✓] Session statistics         : {timings["sample_dask_sessions"]:.2f}s\n')

dask_sample_total = sum(v for k, v in timings.items() if k.startswith('sample_dask_'))
print(f'Dask sample compute total: {dask_sample_total:.2f}s')

# ── Quick Spark tests on sample ─────────────────────────────────────────────
print('\n=== SPARK Tests on Sample ===')
sample_spark_results = {}

if not SPARK_AVAILABLE:
    print('Spark not available — skipping sample Spark tests.')
else:
    from pyspark.sql import functions as F
    
    spark_sample = get_spark_session('EcomPipeline-A7-Sample')
    
    # Save sample as CSV and read with Spark (avoids parquet format incompatibility)
    temp_sample_csv = str(OUTPUT_DIR / 'temp_sample.csv')
    try:
        sample_ddf_computed.to_csv(temp_sample_csv, index=False)
        with log.timer('sample_spark_load') as t:
            sdf_sample = (
                spark_sample.read
                .option('header', 'true')
                .option('inferSchema', 'true')
                .option('timestampFormat', 'yyyy-MM-dd HH:mm:ss z')
                .csv(temp_sample_csv)
            )
        timings['sample_spark_load'] = t.elapsed
        print(f'[✓] Loaded sample to Spark    : {timings["sample_spark_load"]:.2f}s')
        
        # 1 — Revenue by category (sample)
        with log.timer('sample_spark_revenue') as t:
            sample_spark_results['revenue'] = compute_revenue_by_category_spark(sdf_sample)
        timings['sample_spark_revenue'] = t.elapsed
        print(f'[✓] Revenue by category       : {timings["sample_spark_revenue"]:.2f}s')
        
        # 2 — Conversion funnel (sample)
        with log.timer('sample_spark_funnel') as t:
            sample_spark_results['funnel'] = compute_conversion_funnel_spark(sdf_sample)
        timings['sample_spark_funnel'] = t.elapsed
        print(f'[✓] Conversion funnel         : {timings["sample_spark_funnel"]:.2f}s')
        
        # 3 — Window rank (sample)
        with log.timer('sample_spark_window') as t:
            sample_spark_results['window'] = compute_window_rank_spark(sdf_sample)
        timings['sample_spark_window'] = t.elapsed
        print(f'[✓] Window rank (per category): {timings["sample_spark_window"]:.2f}s')
        
        # 4 — Hourly activity (sample)
        with log.timer('sample_spark_hourly') as t:
            sample_spark_results['hourly'] = compute_hourly_activity_spark(sdf_sample)
        timings['sample_spark_hourly'] = t.elapsed
        print(f'[✓] Hourly activity           : {timings["sample_spark_hourly"]:.2f}s\n')
        
        spark_sample_total = sum(v for k, v in timings.items() if k.startswith('sample_spark_'))
        print(f'Spark sample compute total: {spark_sample_total:.2f}s')
    finally:
        spark_sample.stop()
        # Cleanup temp CSV
        if Path(temp_sample_csv).exists():
            Path(temp_sample_csv).unlink()

print('\n=== Sample Processing Complete ===')
print('All transformations ran successfully on the sample dataset.')
print('Ready to proceed with full-scale processing in Stages 2 & 3.')

{"ts": "2026-03-13T10:10:27.553970+00:00", "level": "INFO", "event": "sample_creation_started"}


Creating sample dataset (1.0% of validated data)...


{"ts": "2026-03-13T10:10:40.454017+00:00", "level": "INFO", "event": "sample_creation_completed", "elapsed_s": 12.9}


  Original rows      : 109950731
  Sample rows        : 1,099,506
  Time to create     : 12.9s

=== DASK Tests on Sample ===


{"ts": "2026-03-13T10:10:41.579335+00:00", "level": "INFO", "event": "sample_dask_revenue_started"}
{"ts": "2026-03-13T10:10:42.479282+00:00", "level": "INFO", "event": "revenue_by_category_done", "categories": 14}
{"ts": "2026-03-13T10:10:42.482793+00:00", "level": "INFO", "event": "sample_dask_revenue_completed", "elapsed_s": 0.9}
{"ts": "2026-03-13T10:10:42.482793+00:00", "level": "INFO", "event": "sample_dask_funnel_started"}


...

[✓] Session statistics         : 0.30s

Dask sample compute total: 6.58s

=== SPARK Tests on Sample ===


{"ts": "2026-03-13T10:10:53.935327+00:00", "level": "INFO", "event": "sample_spark_load_started"}
{"ts": "2026-03-13T10:10:56.191150+00:00", "level": "INFO", "event": "sample_spark_load_completed", "elapsed_s": 2.26}
{"ts": "2026-03-13T10:10:56.192533+00:00", "level": "INFO", "event": "sample_spark_revenue_started"}
{"ts": "2026-03-13T10:10:56.313201+00:00", "level": "INFO", "event": "spark_revenue_done"}
{"ts": "2026-03-13T10:10:56.314200+00:00", "level": "INFO", "event": "sample_spark_revenue_completed", "elapsed_s": 0.12}
{"ts": "2026-03-13T10:10:56.314200+00:00", "level": "INFO", "event": "sample_spark_funnel_started"}


[✓] Loaded sample to Spark    : 2.26s
[✓] Revenue by category       : 0.12s


{"ts": "2026-03-13T10:10:57.908513+00:00", "level": "INFO", "event": "spark_funnel_done"}
{"ts": "2026-03-13T10:10:57.909513+00:00", "level": "INFO", "event": "sample_spark_funnel_completed", "elapsed_s": 1.6}
{"ts": "2026-03-13T10:10:57.909513+00:00", "level": "INFO", "event": "sample_spark_window_started"}
{"ts": "2026-03-13T10:10:57.993248+00:00", "level": "INFO", "event": "spark_window_rank_done"}
{"ts": "2026-03-13T10:10:57.994248+00:00", "level": "INFO", "event": "sample_spark_window_completed", "elapsed_s": 0.08}
{"ts": "2026-03-13T10:10:57.994248+00:00", "level": "INFO", "event": "sample_spark_hourly_started"}
{"ts": "2026-03-13T10:10:58.023358+00:00", "level": "INFO", "event": "spark_hourly_done"}
{"ts": "2026-03-13T10:10:58.024861+00:00", "level": "INFO", "event": "sample_spark_hourly_completed", "elapsed_s": 0.03}


[✓] Conversion funnel         : 1.60s
[✓] Window rank (per category): 0.08s
[✓] Hourly activity           : 0.03s

Spark sample compute total: 4.09s

=== Sample Processing Complete ===
All transformations ran successfully on the sample dataset.
Ready to proceed with full-scale processing in Stages 2 & 3.


---
## Stage 2 — Distributed Processing with Dask

Five analyses, each using a distinct operation class:

| # | Analysis | Operations | Scheduler |
|---|---|---|---|
| 1 | Revenue by category | `filter` + `groupby` aggregate | distributed |
| 2 | Conversion funnel | per-event-type `filter` + single-key `groupby` | distributed |
| 3 | Hourly activity | per-event-type `filter` + `assign` + `groupby` | distributed |
| 4 | Session statistics | map-reduce `map_partitions` + pandas re-aggregation | **threaded** (no cluster) |
| 5 | Top brands | `filter` + `nlargest` | distributed |

> **Memory note:** Analysis 4 runs **outside** the distributed cluster because `groupby(user_session)` over millions of unique UUIDs triggers a `repartitiontofewer` step inside Dask's `.compute()` that OOMs workers. Without an active distributed client, the threaded scheduler concatenates partition results directly in the driver process.

In [7]:
import dask

# Allow workers to spill to disk before hitting the kill threshold
dask.config.set({
    'distributed.worker.memory.target':    0.65,
    'distributed.worker.memory.spill':     0.75,
    'distributed.worker.memory.pause':     0.85,
    'distributed.worker.memory.terminate': 0.95,
})

dask_results = {}

# ── Analyses 1, 2, 3, 5 — run inside distributed cluster ────────────────────
with LocalCluster(n_workers=2, threads_per_worker=2, memory_limit='4GB') as cluster:
    with Client(cluster) as client:
        print(f'Dashboard : {client.dashboard_link}')
        print(f'Workers   : {len(client.scheduler_info()["workers"])}\n')

        # 1 — Revenue by category
        with log.timer('dask_revenue') as t:
            dask_results['revenue'] = compute_revenue_by_category(validated_ddf)
        timings['dask_revenue'] = t.elapsed
        print('--- Top 10 Categories by Revenue ---')
        print(dask_results['revenue'].head(10).to_string(index=False))

        # 2 — Conversion funnel
        with log.timer('dask_funnel') as t:
            dask_results['funnel'] = compute_conversion_funnel(validated_ddf)
        timings['dask_funnel'] = t.elapsed
        cols = [c for c in ['top_category','view','cart','purchase','purchase_rate','cart_rate']
                if c in dask_results['funnel'].columns]
        print('\n--- Conversion Funnel (top 5 by purchase rate) ---')
        print(dask_results['funnel'].sort_values('purchase_rate', ascending=False).head(5)[cols].to_string(index=False))

        # 3 — Hourly activity
        with log.timer('dask_hourly') as t:
            dask_results['hourly'] = compute_hourly_activity(validated_ddf)
        timings['dask_hourly'] = t.elapsed
        print('\n--- Events by Hour (purchases) ---')
        hourly_purchase = dask_results['hourly'][dask_results['hourly']['event_type'] == 'purchase']
        for _, row in hourly_purchase.iterrows():
            bar = '#' * (int(row['count']) // 50_000)
            print(f'  {int(row["hour"]):2d}:00  {int(row["count"]):>10,}  {bar}')

        # 5 — Top brands
        with log.timer('dask_brands') as t:
            dask_results['brands'] = compute_top_brands(validated_ddf, top_n=20)
        timings['dask_brands'] = t.elapsed
        print('\n--- Top 10 Brands by Purchase Revenue ---')
        print(dask_results['brands'].head(10).to_string(index=False))

# ── Analysis 4: Session stats — no active distributed client ─────────────────
print('\n--- Session Analysis (threaded scheduler, no distributed client) ---')
with log.timer('dask_sessions') as t:
    dask_results['sessions'] = compute_session_stats(validated_ddf)
timings['dask_sessions'] = t.elapsed
sess = dask_results['sessions']
print(f'  Total sessions      : {len(sess):,}')
print(f'  Avg events/session  : {sess["num_events"].mean():.1f}')
print(f'  Median duration     : {sess["session_duration_min"].median():.1f} min')
print(f'  Avg spend/session   : {sess["total_spend"].mean():.2f}')

print(f'\nDask compute total: {sum(v for k,v in timings.items() if k.startswith("dask_")):.1f}s')

{"ts": "2026-03-13T10:13:27.856927+00:00", "level": "INFO", "event": "dask_revenue_started"}


Dashboard : http://127.0.0.1:8787/status
Workers   : 2



2026-03-13 11:13:28,499 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle a7a0ffb09eb414e4b26fdb73ffb9a4d5 initialized by task ('shuffle-transfer-a7a0ffb09eb414e4b26fdb73ffb9a4d5', 31) executed on worker tcp://127.0.0.1:57382
2026-03-13 11:13:35,123 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle a7a0ffb09eb414e4b26fdb73ffb9a4d5 deactivated due to stimulus 'task-finished-1773396815.1225166'
{"ts": "2026-03-13T10:13:35.142805+00:00", "level": "INFO", "event": "revenue_by_category_done", "categories": 14}
{"ts": "2026-03-13T10:13:35.143805+00:00", "level": "INFO", "event": "dask_revenue_completed", "elapsed_s": 7.29}
{"ts": "2026-03-13T10:13:35.144964+00:00", "level": "INFO", "event": "dask_funnel_started"}


--- Top 10 Categories by Revenue ---
top_category  total_revenue  num_purchases  avg_price
 electronics   381714286.57         916667 416.415434
     unknown    52805443.81         407643 129.538454
  appliances    32223623.59         174022 185.169827
   computers    25373205.34          62332 407.065477
   furniture     4216980.01          19843 212.517261
        auto     2649036.47          21339 124.140610
construction     2013385.72          16500 122.023377
     apparel     1805962.87          22217  81.287432
        kids     1401903.93          11648 120.355763
       sport      718251.75           2725 263.578624


2026-03-13 11:13:35,631 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 47a0334fa21a48bebcdef0c1a3b5fa3a initialized by task ('shuffle-transfer-47a0334fa21a48bebcdef0c1a3b5fa3a', 26) executed on worker tcp://127.0.0.1:57382
2026-03-13 11:14:52,928 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 47a0334fa21a48bebcdef0c1a3b5fa3a deactivated due to stimulus 'task-finished-1773396892.9258547'
2026-03-13 11:14:53,304 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 349dc838870d6b5f5bf00b1ae1ede655 initialized by task ('shuffle-transfer-349dc838870d6b5f5bf00b1ae1ede655', 41) executed on worker tcp://127.0.0.1:57382
2026-03-13 11:16:16,499 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 349dc838870d6b5f5bf00b1ae1ede655 deactivated due to stimulus 'task-finished-1773396976.4997213'
2026-03-13 11:16:16,847 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 0684a684ca8f6ca49d75564a5eeeccba initialized by task ('shuffle-transfer-0684a684ca8f

...


--- Top 10 Brands by Purchase Revenue ---
  brand  total_revenue  num_purchases
  apple   238721793.70         308937
samsung   101277413.48         372923
 xiaomi    20453899.25         124908
 huawei     9664104.09          47204
     lg     8626906.72          21606
   acer     6924026.05          13284
lucente     6651658.94          26137
   sony     6341082.98          17038
   oppo     5901500.52          25971
 lenovo     4450744.83          11125


{"ts": "2026-03-13T10:20:00.754408+00:00", "level": "INFO", "event": "dask_sessions_started"}
{"ts": "2026-03-13T10:20:00.754408+00:00", "level": "INFO", "event": "session_stats_phase1_started"}



--- Session Analysis (threaded scheduler, no distributed client) ---


{"ts": "2026-03-13T10:20:14.558536+00:00", "level": "INFO", "event": "session_stats_phase1_done", "partial_rows": 27106776}
{"ts": "2026-03-13T10:20:22.038189+00:00", "level": "INFO", "event": "session_stats_done", "sessions": 23016650}
{"ts": "2026-03-13T10:20:22.069819+00:00", "level": "INFO", "event": "dask_sessions_completed", "elapsed_s": 21.32}


  Total sessions      : 23,016,650
  Avg events/session  : 4.8
  Median duration     : 1.0 min
  Avg spend/session   : 1393.14

Dask compute total: 413.7s


In [8]:
# ── Persist Dask results as Parquet ───────────────────────────────────────────
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

save_parquet_pandas(dask_results['revenue'],                RESULTS_DIR / 'dask_revenue.parquet')
save_parquet_pandas(dask_results['funnel'],                 RESULTS_DIR / 'dask_funnel.parquet')
save_parquet_pandas(dask_results['hourly'],                 RESULTS_DIR / 'dask_hourly.parquet')
save_parquet_pandas(dask_results['brands'],                 RESULTS_DIR / 'dask_brands.parquet')
save_parquet_pandas(dask_results['sessions'].head(500_000), RESULTS_DIR / 'dask_sessions.parquet')

print('Dask results saved to:', RESULTS_DIR)

{"ts": "2026-03-13T10:32:42.976396+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "F:\\Group Project-A7\\Group-A7-Project\\output\\results\\dask_revenue.parquet", "rows": 14}
{"ts": "2026-03-13T10:32:42.978398+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "F:\\Group Project-A7\\Group-A7-Project\\output\\results\\dask_funnel.parquet", "rows": 14}
{"ts": "2026-03-13T10:32:42.979398+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "F:\\Group Project-A7\\Group-A7-Project\\output\\results\\dask_hourly.parquet", "rows": 72}
{"ts": "2026-03-13T10:32:42.981906+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "F:\\Group Project-A7\\Group-A7-Project\\output\\results\\dask_brands.parquet", "rows": 20}
{"ts": "2026-03-13T10:32:43.178208+00:00", "level": "INFO", "event": "saved_parquet_pandas", "path": "F:\\Group Project-A7\\Group-A7-Project\\output\\results\\dask_sessions.parquet", "rows": 500000}


Dask results saved to: F:\Group Project-A7\Group-A7-Project\output\results


---
## Stage 3 — Distributed Processing with Apache Spark

Same analytical questions answered via PySpark for framework comparison.  
Spark reads **directly from the raw CSV files** to avoid Parquet timestamp format incompatibilities between pyarrow (Dask) and the JVM Parquet reader (Spark).  
**Bonus**: analysis 3 uses a **window function** (`RANK() OVER PARTITION BY`) to rank products within each category.

> If PySpark is not installed, this stage is skipped automatically.

In [ ]:
spark_results = {}

if not SPARK_AVAILABLE:
    print('Spark not available — skipping Stage 3.')
else:
    from pyspark.sql import functions as F

    spark = get_spark_session('EcomPipeline-A7')
    print(f'Dashboard    : {get_spark_dashboard_url(spark)}')
    print(f'Spark version: {spark.version}\n')

    # Spark reads the raw CSVs directly — avoids pyarrow/JVM Parquet incompatibility
    # Use forward-slash paths (Spark on Windows requires this)
    csv_paths = [
        str(RAW_OCT).replace('\\', '/'),
        str(RAW_NOV).replace('\\', '/'),
    ]
    with log.timer('spark_load') as t:
        sdf_raw = load_csv_spark(spark, csv_paths)
    timings['spark_load'] = t.elapsed

    # Apply the same cleaning filters as Stage 1
    valid_events = list(VALID_EVENT_TYPES)
    sdf = (
        sdf_raw
        .dropna(subset=['user_id', 'product_id', 'price', 'event_time', 'user_session'])
        .filter(F.col('event_type').isin(valid_events))
        .filter(F.col('price') >= 0)
        .filter(F.trim(F.col('user_session')) != '')
    )
    print(f'Spark partitions after cleaning: {sdf.rdd.getNumPartitions()}')
    sdf.printSchema()

    # 1 — Revenue by category
    with log.timer('spark_revenue') as t:
        spark_results['revenue'] = compute_revenue_by_category_spark(sdf)
    timings['spark_revenue'] = t.elapsed
    print('--- Spark: Top 10 Categories by Revenue ---')
    spark_results['revenue'].show(10, truncate=False)

    # 2 — Conversion funnel
    with log.timer('spark_funnel') as t:
        spark_results['funnel'] = compute_conversion_funnel_spark(sdf)
    timings['spark_funnel'] = t.elapsed
    print('--- Spark: Conversion Funnel ---')
    spark_results['funnel'].orderBy('purchase_rate', ascending=False).show(10, truncate=False)

    # 3 — Window function: rank products within category
    with log.timer('spark_window') as t:
        spark_results['window'] = compute_window_rank_spark(sdf)
    timings['spark_window'] = t.elapsed
    print('--- Spark: Top Products per Category (window rank) ---')
    spark_results['window'].show(20, truncate=False)

    # 4 — Hourly activity
    with log.timer('spark_hourly') as t:
        spark_results['hourly'] = compute_hourly_activity_spark(sdf)
    timings['spark_hourly'] = t.elapsed

    # Persist Spark results as CSV (Parquet has Windows/JVM compatibility issues)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    results_base = RESULTS_DIR.as_posix()
    
    print('\n--- Persisting Spark results to CSV ---')
    spark_results['revenue'].coalesce(1).write.mode('overwrite').option('header', 'true').csv(f'{results_base}/spark_revenue')
    print('[✓] Revenue')
    spark_results['funnel'].coalesce(1).write.mode('overwrite').option('header', 'true').csv(f'{results_base}/spark_funnel')
    print('[✓] Funnel')
    spark_results['window'].coalesce(1).write.mode('overwrite').option('header', 'true').csv(f'{results_base}/spark_window')
    print('[✓] Window rank')
    spark_results['hourly'].coalesce(1).write.mode('overwrite').option('header', 'true').csv(f'{results_base}/spark_hourly')
    print('[✓] Hourly activity')

    print(f'\nSpark compute total: {sum(v for k,v in timings.items() if k.startswith("spark_")):.1f}s')
    spark.stop()

{"ts": "2026-03-13T10:42:23.664714+00:00", "level": "INFO", "event": "spark_load_started"}


Dashboard    : http://host.docker.internal:4040
Spark version: 4.1.1



{"ts": "2026-03-13T10:43:05.228920+00:00", "level": "INFO", "event": "spark_csv_loaded", "files": 2, "partitions": 110}
{"ts": "2026-03-13T10:43:05.228920+00:00", "level": "INFO", "event": "spark_load_completed", "elapsed_s": 41.56}
{"ts": "2026-03-13T10:43:05.261625+00:00", "level": "INFO", "event": "spark_revenue_started"}
{"ts": "2026-03-13T10:43:05.283212+00:00", "level": "INFO", "event": "spark_revenue_done"}
{"ts": "2026-03-13T10:43:05.283749+00:00", "level": "INFO", "event": "spark_revenue_completed", "elapsed_s": 0.02}


Spark partitions after cleaning: 110
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)

--- Spark: Top 10 Categories by Revenue ---


{"ts": "2026-03-13T10:43:46.760947+00:00", "level": "INFO", "event": "spark_funnel_started"}


...

{"ts": "2026-03-13T10:45:10.112380+00:00", "level": "INFO", "event": "spark_window_started"}
{"ts": "2026-03-13T10:45:10.147159+00:00", "level": "INFO", "event": "spark_window_rank_done"}
{"ts": "2026-03-13T10:45:10.147159+00:00", "level": "INFO", "event": "spark_window_completed", "elapsed_s": 0.03}


+------------+-------+--------+--------+--------------------+--------------------+
|top_category|cart   |purchase|view    |purchase_rate       |cart_rate           |
+------------+-------+--------+--------+--------------------+--------------------+
|electronics |2198451|916667  |37026582|0.024756997553811475|0.05937493771366744 |
|medicine    |1796   |654     |34738   |0.018826645172433647|0.05170130692613276 |
|stationery  |750    |325     |19323   |0.016819334471872897|0.03881384878124515 |
|appliances  |445181 |174022  |12837916|0.013555315364269403|0.034677045713650094|
|unknown     |932216 |407643  |34073918|0.011963490667553993|0.027358638357937   |
|computers   |145266 |62332   |6297977 |0.009897146337625558|0.023065501827015247|
|auto        |48229  |21339   |2157706 |0.009889669862344545|0.022351979370683495|
|construction|46727  |16500   |1759762 |0.009376267927140147|0.026553022510998645|
|kids        |23353  |11648   |1292002 |0.009015465920331393|0.018075049419428144|
|spo

{"ts": "2026-03-13T10:45:51.860813+00:00", "level": "INFO", "event": "spark_hourly_started"}
{"ts": "2026-03-13T10:45:51.873270+00:00", "level": "INFO", "event": "spark_hourly_done"}
{"ts": "2026-03-13T10:45:51.874275+00:00", "level": "INFO", "event": "spark_hourly_completed", "elapsed_s": 0.01}


+------------+----------+------------------+---------+----------------+
|top_category|product_id|product_revenue   |num_sales|rank_in_category|
+------------+----------+------------------+---------+----------------+
|accessories |28401112  |2256.9999999999995|56       |1               |
|accessories |28401058  |2007.8000000000009|20       |2               |
|accessories |28400759  |1706.2999999999995|50       |3               |
|accessories |28400912  |1330.1700000000005|31       |4               |
|accessories |28300432  |1250.9099999999996|27       |5               |
|apparel     |28716666  |7983.890000000001 |85       |1               |
|apparel     |28720716  |7981.020000000002 |101      |2               |
|apparel     |28716978  |7519.3300000000045|91       |3               |
|apparel     |54900004  |7490.339999999998 |97       |4               |
|apparel     |28703609  |6261.949999999999 |53       |5               |
|appliances  |3600661   |974933.7500000001 |3224     |1         

Py4JJavaError: An error occurred while calling o764.csv.
: java.util.concurrent.ExecutionException: Boxed Exception
	at scala.concurrent.impl.Promise$.scala$concurrent$impl$Promise$$resolve(Promise.scala:99)
	at scala.concurrent.impl.Promise$DefaultPromise.tryComplete(Promise.scala:288)
	at scala.concurrent.Promise.complete(Promise.scala:57)
	at scala.concurrent.Promise.complete$(Promise.scala:56)
	at scala.concurrent.impl.Promise$DefaultPromise.complete(Promise.scala:104)
	at scala.concurrent.Promise.failure(Promise.scala:109)
	at scala.concurrent.Promise.failure$(Promise.scala:109)
	at scala.concurrent.impl.Promise$DefaultPromise.failure(Promise.scala:104)
	at org.apache.spark.sql.execution.adaptive.ResultQueryStageExec.$anonfun$doMaterialize$2(QueryStageExec.scala:336)
	at java.base/java.util.concurrent.CompletableFuture.uniWhenComplete(CompletableFuture.java:863)
	at java.base/java.util.concurrent.CompletableFuture$UniWhenComplete.tryFire(CompletableFuture.java:841)
	at

---
## Stage 4 — Results Summary & Framework Comparison

### Storage Strategy

| Layer | Format | Location | Why |
|---|---|---|---|
| Raw input | CSV | `data/` | Source of truth, unchanged |
| Validated intermediate | **Parquet** (partitioned by `event_type`) | `output/parquet/validated/` | Predicate pushdown speeds downstream reads |
| Analysis results | **Parquet** (single file) | `output/results/` | Compact, typed, directly loadable by BI tools |

In [11]:
# ── Timing comparison ─────────────────────────────────────────────────────────
print('=== Pipeline Stage Timings ===')
print(f'{"Stage":<25} {"Time (s)":>10}')
print('-' * 37)
for stage, elapsed in timings.items():
    print(f'{stage:<25} {elapsed:>10.1f}')
print('=' * 37)
print(f'{"TOTAL":<25} {sum(timings.values()):>10.1f}')

# ── Output files ──────────────────────────────────────────────────────────────
print('\n=== Output Files ===')
for p in sorted(Path(OUTPUT_DIR).rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / 1_048_576
        print(f'  {str(p.relative_to(PROJECT_ROOT)):<55} {size_mb:>7.2f} MB')

# ── Framework notes ───────────────────────────────────────────────────────────
print("""
=== Framework Comparison Notes ===
Dask
  + Familiar pandas-like API, easy iteration in notebooks
  + Native Python, no JVM overhead
  + Low-latency scheduler startup
  - High-cardinality groupby OOMs via repartitiontofewer in distributed mode;
    must run outside Client context with threaded scheduler
  - Multi-key shuffles require per-key decomposition to avoid OOM

Spark
  + Catalyst optimiser produces efficient physical plans
  + First-class window functions, complex joins
  + Graceful spill-to-disk for large shuffles
  + No format interoperability issues (reads CSV directly)
  - JVM startup adds ~30s latency
  - More verbose API; Windows path handling quirks

Verdict: Dask is faster for simple aggregations on a single node.
Spark is more robust for complex operations (windows, multi-key joins)
and scales better on managed clusters (EMR, Dataproc).
""")

=== Pipeline Stage Timings ===
Stage                       Time (s)
-------------------------------------
ingestion                        0.1
cleaning                         0.0
save_validated                 273.8
sample_creation                 12.9
sample_dask_revenue              0.9
sample_dask_funnel               3.5
sample_dask_hourly               1.4
sample_dask_brands               0.4
sample_dask_sessions             0.3
sample_spark_load                2.3
sample_spark_revenue             0.1
sample_spark_funnel              1.6
sample_spark_window              0.1
sample_spark_hourly              0.0
dask_revenue                     7.3
dask_funnel                    339.8
dask_hourly                     35.1
dask_brands                     10.2
dask_sessions                   21.3
spark_load                      41.6
spark_revenue                    0.0
spark_funnel                    41.6
spark_window                     0.0
spark_hourly                     0.0
TOTAL 